# 02 · Train, evaluate and register the litigation model

**The business question:** when a new workers' comp claim comes in, how likely is it that the claimant ends up
with an attorney or representative? Litigated claims usually cost more and stay open longer, so a score at intake
can help route claims to experienced adjusters early.

**What kind of model this is.** This is *supervised learning*. We show the model hundreds of thousands of past
claims where we already know the answer (litigated or not), and it learns patterns that connect claim details to
that answer. The answer is yes/no, so this is a *classification* model. For each claim it outputs a
**probability** between 0 and 1.

### Key terms

| Term | Meaning in this notebook |
|---|---|
| **Feature** | An input the model uses: age at injury, body part injured, industry, and so on |
| **Label** (target) | The answer being predicted: `IS_LITIGATED` (1 = attorney/representative involved, 0 = not) |
| **Training set** | Claims the model learns from (assembled 2017–2020) |
| **Validation set** | Claims used to tune settings and compare models (2021) |
| **Test set** | Claims left untouched until the very end, for an honest final grade (2022) |
| **Overfitting** | When a model memorizes quirks of its training data and does worse on new data. Validation and test sets catch this. |
| **Pipeline** | Data-preparation steps and the model bundled into one object, so new data always gets identical preparation |

### Steps

1. Connect to Snowflake and set options
2. Load the data and split it by time
3. Baseline model: logistic regression
4. Main model: XGBoost
5. Final grade on the untouched test year
6. Register the model in Snowflake

### Where does each piece of code run?

Three parts of Snowflake are involved:

* **The notebook service** (the compute you connected to) runs the Python: pandas, scikit-learn and XGBoost.
  Model training happens here.
* **The warehouse `WC_ML_WH`** runs SQL. Whenever the code builds a query (`session.table(...)`,
  `session.sql(...)`), the work happens in the warehouse and only the result comes back.
* **The Model Registry** (schema `WC_ML.MODELS`) stores the trained model as a Snowflake object that both SQL and
  Python can call.

Notes marked **Snowflake vs. local** explain what would be different if you ran this on your own laptop.
In short:

| Task | In Snowflake (this notebook) | On a laptop |
|---|---|---|
| Connect | `get_active_session()`: you're already logged in | Build a session with your account, user and password, or skip Snowflake and read files with pandas |
| Prepare features | The SQL view `WC_ML.ANALYTICS.CLAIM_FEATURES` (`sql/03_features.sql`) | The same logic rewritten in pandas |
| Load data | A Snowpark query runs in the warehouse, and `to_pandas()` brings the result back | `pd.read_parquet(...)` loads every file into laptop memory |
| Train | Same scikit-learn / XGBoost code, on the notebook service's CPU | Same code, on your laptop's CPU |
| Save the model | `Registry.log_model(...)` creates a versioned Snowflake object with permissions | `joblib.dump(model, "model.pkl")` writes a file you manage yourself |
| Use the model | From SQL (`mv!PREDICT_PROBA`) or Python (`mv.run`), next to the data | Load the file in Python, move the data to it, then call `predict_proba` |

**The modeling code itself is identical in both places.** What changes is how you connect, where the data comes
from, and how the trained model is stored and served.

## 1. Connect to Snowflake and set options

In [ ]:
# --- Imports ----------------------------------------------------------------------------------
# Snowpark is Snowflake's Python library. It lets Python code build queries that run inside Snowflake.
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark import functions as F   # column helpers, e.g. F.col("ASSEMBLY_YEAR") in a filter
import pandas as pd                             # pandas: the standard Python library for in-memory tables

# --- Connect ----------------------------------------------------------------------------------
# [Snowflake vs. local] Inside a Snowflake notebook you're already logged in, so get_active_session()
# just hands back that connection. On a laptop you'd create it yourself, for example:
#     from snowflake.snowpark import Session
#     session = Session.builder.configs({
#         "account": "<account id>", "user": "<user>", "password": "<password>",
#         "role": "WC_ML_ROLE", "warehouse": "WC_ML_WH",
#     }).create()
session = get_active_session()

# All SQL sent from this notebook runs on this warehouse. It bills per second while it's busy
# and suspends itself after 60 idle seconds.
session.use_warehouse("WC_ML_WH")

# Show up to 50 columns when a pandas table is displayed
pd.set_option("display.max_columns", 50)


def q(sql: str) -> pd.DataFrame:
    """Run a SQL statement in Snowflake and return the result as a pandas table."""
    return session.sql(sql).to_pandas()


# Sanity check: should show WC_ML_ROLE and WC_ML_WH.
# (In a notebook, the value of a cell's last line is displayed automatically.)
q("SELECT CURRENT_ROLE() AS ROLE, CURRENT_WAREHOUSE() AS WAREHOUSE")

In [ ]:
# --- Installed package versions -------------------------------------------------------------
# [Snowflake vs. local] Container Runtime comes with these preinstalled. On a laptop you'd run
#     pip install snowflake-ml-python scikit-learn xgboost
from importlib.metadata import PackageNotFoundError, version

def installed(pkg):
    """Return the installed version of a package, or NOT INSTALLED."""
    try:
        return version(pkg)
    except PackageNotFoundError:
        return "NOT INSTALLED"

print({pkg: installed(pkg) for pkg in ["snowflake-ml-python", "scikit-learn", "xgboost"]})

# --- Settings ---------------------------------------------------------------------------------
# The cleaned feature view built by sql/03_features.sql. Training AND scoring both read this view,
# so the model always sees data prepared in exactly the same way.
FEATURE_VIEW = "WC_ML.ANALYTICS.CLAIM_FEATURES"
# The name the model will have in the Model Registry
MODEL_NAME = "WC_LITIGATION_XGB"
# The column to predict: 1 = the claimant had an attorney/representative, 0 = not
LABEL = "IS_LITIGATED"

# Features (model inputs), split by type because each type is prepared differently.
# - Numeric: numbers where bigger or smaller means something (age, days from accident to report).
# - Categorical: codes or labels where the value names a group rather than an amount
#   (body-part code 42 is not "twice" code 21).
# Order matters: the registered model expects its inputs in exactly this order, and the SQL in
# deploy/04_deploy_scoring.sql passes them in this order.
NUMERIC = ["AGE_AT_INJURY", "REPORT_LAG_DAYS", "ACCIDENT_MONTH", "ACCIDENT_DOW"]
CATEGORICAL = ["GENDER", "CARRIER_TYPE", "DISTRICT_NAME", "COUNTY_OF_INJURY", "MEDICAL_FEE_REGION",
               "INDUSTRY_CODE", "WCIO_CAUSE_CODE", "WCIO_NATURE_CODE", "WCIO_BODY_CODE", "COVID_19_INDICATOR"]
FEATURES = NUMERIC + CATEGORICAL

# Split by TIME, not at random: learn from older claims, check on newer ones.
# This mirrors real use, where a model trained on the past scores claims that arrive later.
TRAIN_YEARS = (2017, 2020)   # learn from these years
VALID_YEAR = 2021            # tune settings and compare models on this year
TEST_YEAR = 2022             # final, untouched check

# Cap on training rows. A random 600k claims is plenty for this model and keeps memory use
# and run time modest on the small notebook machine.
MAX_TRAIN_ROWS = 600_000
# A fixed "random seed" makes sampling and training repeatable: same inputs, same results.
RANDOM_STATE = 42

## 2. Load the data and split it by time

`session.table(...)` returns a **Snowpark DataFrame**. It looks like pandas but is *lazy*: each `.filter()` or
`.select()` only adds to a SQL query, and nothing runs until you ask for results. `to_pandas()` sends the query to
the warehouse and copies the result into the notebook's memory as an ordinary pandas DataFrame. From that point,
all the work happens on the notebook service.

**Why split by time?** A random split would mix 2022 claims into training and then test on other 2022 claims,
which is easier than real life. Training on 2017–2020 and testing on 2022 asks the real question: does what we
learned from the past still work on later claims?

> **Snowflake vs. local:** locally you'd load files with `pd.read_parquet("data/ny_wcb/")` and redo the
> feature logic from `sql/03_features.sql` in pandas. Here the warehouse does the filtering and feature work, and
> only the needed rows and columns travel to Python.

In [ ]:
# Build the query step by step. Nothing runs until .to_pandas().
df = (
    session.table(FEATURE_VIEW)                                          # FROM WC_ML.ANALYTICS.CLAIM_FEATURES
    .filter(F.col("ASSEMBLY_YEAR").between(TRAIN_YEARS[0], TEST_YEAR))  # WHERE ASSEMBLY_YEAR BETWEEN 2017 AND 2022
    .select("CLAIM_IDENTIFIER", "ASSEMBLY_YEAR", LABEL, *FEATURES)     # SELECT only the columns needed
    .to_pandas()                                                         # run it, then copy the result into memory
)

# Make column types explicit, so the model and the registry's input "signature" see consistent types.
df[NUMERIC] = df[NUMERIC].astype("float64")    # decimal numbers; missing values become NaN ("not a number")
df[CATEGORICAL] = df[CATEGORICAL].astype(str)  # text categories
df[LABEL] = df[LABEL].astype(int)              # 0 or 1

# --- Time-based split ---------------------------------------------------------------------------
train_df = df[df.ASSEMBLY_YEAR.between(*TRAIN_YEARS)]
if len(train_df) > MAX_TRAIN_ROWS:
    # A random sample keeps training fast. The share of litigated claims stays about the same.
    train_df = train_df.sample(MAX_TRAIN_ROWS, random_state=RANDOM_STATE)
valid_df = df[df.ASSEMBLY_YEAR == VALID_YEAR]
test_df = df[df.ASSEMBLY_YEAR == TEST_YEAR]

# By convention X holds the features (inputs) and y holds the label (the answer).
X_train, y_train = train_df[FEATURES], train_df[LABEL]
X_valid, y_valid = valid_df[FEATURES], valid_df[LABEL]
X_test, y_test = test_df[FEATURES], test_df[LABEL]

# Summary: rows per set and each set's litigation rate (also called the "base rate").
pd.DataFrame({
    "rows": [len(train_df), len(valid_df), len(test_df)],
    "litigation_rate": [y_train.mean(), y_valid.mean(), y_test.mean()],
}, index=["train", "valid", "test"]).round(4)

**Reading the output:** expect the litigation rate to drift a little between years. Later years can look slightly
lower because some claims haven't had time to pick up an attorney yet. Every model below is graded against these
base rates.

## 3. Baseline model: logistic regression

Start with a simple, well-understood model. If the fancier model can't clearly beat it, the fancier model isn't
worth its complexity.

**Logistic regression** gives each feature a weight, adds up the weighted values, and converts the total into a
probability. Before it can do that, the data needs preparing:

* **Imputation:** fills missing numbers (for example an unknown age) with the median, because this model can't
  handle blanks.
* **Scaling:** puts the numeric features on a similar scale, so age (about 20–70) doesn't swamp month (1–12).
* **One-hot encoding:** turns each category into its own 0/1 column (`CARRIER_TYPE = Private` becomes a column
  that is 1 for private carriers and 0 otherwise). `min_frequency=50` groups very rare categories together, and
  categories never seen in training go into that same group (or get all zeros if no group was needed), so new
  codes don't break the model.

### How models are graded

| Metric | What it tells you | Good direction |
|---|---|---|
| **ROC-AUC** | Pick a random litigated claim and a random non-litigated claim: how often does the litigated one get the higher score? 0.5 is a coin flip, 1.0 is perfect. Around 0.7 is useful and 0.8+ is strong for a problem like this. | Higher |
| **PR-AUC** (average precision) | How well the model finds litigated claims without many false alarms. A model with no skill scores about the base rate, so compare it to that. | Higher |
| **Brier score** | The average squared gap between the predicted probability and what actually happened (0 or 1). It rewards probabilities you can take at face value. | Lower |
| **base_rate** | The share of claims that were litigated: the bar to beat | n/a |

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler


def evaluate(name, y, p):
    """Grade predicted probabilities p against true answers y (see the metrics table above)."""
    return {"model": name, "roc_auc": roc_auc_score(y, p), "pr_auc": average_precision_score(y, p),
            "brier": brier_score_loss(y, p), "base_rate": y.mean()}


# A Pipeline chains the preparation steps and the model into one object:
# .fit() learns the preparation AND the model from training data;
# .predict_proba() applies the exact same preparation to new data before predicting.
logit = Pipeline([
    ("prep", ColumnTransformer([
        # numeric columns: fill blanks with the median, then scale
        ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), NUMERIC),
        # categorical columns: one 0/1 column per category; rare and unseen categories share an "infrequent" column
        ("cat", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=50), CATEGORICAL),
    ])),
    # max_iter raises the number of solver steps allowed, so the fit has room to converge
    ("model", LogisticRegression(max_iter=2000)),
])
logit.fit(X_train, y_train)   # learn from the training years only

# predict_proba returns two columns, P(not litigated) and P(litigated); [:, 1] keeps the second.
results = [evaluate("logistic (valid)", y_valid, logit.predict_proba(X_valid)[:, 1])]
pd.DataFrame(results).round(4)

## 4. Main model: XGBoost

**How it works.** XGBoost builds hundreds of small **decision trees**. A tree is a flowchart of yes/no questions,
for example "Is the report lag over 30 days? Is the body part the back?", ending in a score. The trees are built
one at a time, and each new tree focuses on the claims the earlier trees got most wrong. This is called
*gradient boosting*. The final prediction combines all the trees. Trees pick up interactions automatically, such
as "a long report lag matters more for back injuries", which logistic regression can't do without extra work.

**Data preparation is simpler for trees:**
* **Ordinal encoding:** each category becomes a number (Private = 0, Self-insured = 1, ...). Trees can split on
  those numbers without one-hot columns. Categories never seen in training get −1.
* **Missing values stay as NaN.** XGBoost learns which branch blanks should follow, so no imputation is needed.
* **No scaling:** a tree's yes/no questions don't care about scale.

**Early stopping:** the model keeps adding trees while the score on the 2021 validation year improves. It stops
after 50 trees in a row bring no improvement and keeps the best point. This is the main protection against
overfitting.

| Setting | Value | Meaning |
|---|---|---|
| `n_estimators` | 1000 | Maximum number of trees (early stopping usually ends sooner) |
| `learning_rate` | 0.05 | How much each new tree may change the prediction. Smaller is slower but steadier. |
| `max_depth` | 6 | Maximum number of questions in a row inside one tree. Deeper trees capture more complex patterns but overfit more easily. |
| `min_child_weight` | 20 | Stops the tree from making a rule that applies to only a handful of claims |
| `subsample` / `colsample_bytree` | 0.8 | Each tree sees a random 80% of rows and features, which reduces overfitting |
| `eval_metric` | `aucpr` | The validation score watched during early stopping (PR-AUC) |

In [ ]:
from xgboost import XGBClassifier

# Preparation for the tree model: numbers pass through untouched; categories become integer codes.
prep = ColumnTransformer([
    ("num", "passthrough", NUMERIC),
    ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), CATEGORICAL),
])
prep.fit(X_train)   # learn the category-to-number mapping from training data only

xgb = XGBClassifier(
    n_estimators=1000,         # upper limit on the number of trees
    learning_rate=0.05,        # small steps: more trees, steadier learning
    max_depth=6,               # at most 6 questions in a row per tree
    min_child_weight=20,       # no rules that rest on a tiny group of claims
    subsample=0.8,             # each tree sees a random 80% of claims...
    colsample_bytree=0.8,      # ...and a random 80% of features
    tree_method="hist",        # fast training method for large tables
    eval_metric="aucpr",       # score watched on the validation year
    early_stopping_rounds=50,  # stop after 50 trees with no improvement
    random_state=RANDOM_STATE,
    n_jobs=-1,                 # use every CPU core on the notebook machine
)
# Train on 2017-2020, watching 2021 for early stopping. verbose=100 prints progress every 100 trees.
xgb.fit(prep.transform(X_train), y_train,
        eval_set=[(prep.transform(X_valid), y_valid)], verbose=100)

# Bundle preparation + model into one Pipeline: raw feature columns in, probabilities out.
# This single object is what gets registered and deployed.
xgb_pipe = Pipeline([("prep", prep), ("model", xgb)])

results.append(evaluate("xgboost (valid)", y_valid, xgb_pipe.predict_proba(X_valid)[:, 1]))
pd.DataFrame(results).round(4)

## 5. Final grade on the untouched test year

The 2021 validation year has been used to choose settings, so its scores are slightly optimistic. The 2022 test
year hasn't been used at all, so it gives the honest estimate of performance on new claims. Look at it once, at
the end. If you go back and adjust the model after seeing test results, the test year stops being an honest
check.

In [ ]:
# Probabilities for every 2022 claim from both models
p_test_logit = logit.predict_proba(X_test)[:, 1]
p_test_xgb = xgb_pipe.predict_proba(X_test)[:, 1]

test_metrics = pd.DataFrame([
    evaluate("logistic (test)", y_test, p_test_logit),
    evaluate("xgboost (test)", y_test, p_test_xgb),
]).round(4)
test_metrics

### Lift by risk decile: the business view

Sort the 2022 claims from highest to lowest predicted risk and cut them into 10 equal groups (*deciles*).
Decile 1 is the riskiest 10%.

* **actual_rate:** the share of claims in that decile that really were litigated
* **lift:** the decile's actual rate divided by the overall rate. A lift of 2.5 means claims in that decile were
  litigated 2.5× as often as the average claim.
* **cum_share_of_litigated:** the share of all litigated claims caught by this decile and the ones above it. For
  example, 0.45 at decile 2 means that reviewing the top 20% of claims would catch 45% of eventual litigation.

This is the table to show a claims audience. It turns the model into a concrete triage rule.

In [ ]:
lift = pd.DataFrame({"p": p_test_xgb, "y": y_test.values})
# Rank claims by predicted probability (1 = riskiest), then cut the ranks into 10 equal groups
lift["decile"] = pd.qcut(lift.p.rank(method="first", ascending=False), 10, labels=range(1, 11))

lift_table = lift.groupby("decile", observed=True).agg(
    n=("y", "size"),              # claims per decile
    avg_pred=("p", "mean"),       # average predicted probability
    actual_rate=("y", "mean"),    # share actually litigated
)
lift_table["lift"] = lift_table.actual_rate / lift.y.mean()
lift_table["cum_share_of_litigated"] = lift.groupby("decile", observed=True).y.sum().cumsum() / lift.y.sum()
lift_table.round(3)

### Which features the model relies on

**Gain** measures how much each feature improved the trees' predictions, summed over every question that used it.
It shows what the model *uses*, not what *causes* litigation. Two related features can also share credit
between them.

In [ ]:
# XGBoost names features f0, f1, ... in the order the preparation step outputs them,
# which is NUMERIC followed by CATEGORICAL, the same as FEATURES. Map them back to real names.
gain = pd.Series(xgb.get_booster().get_score(importance_type="gain"))
gain.index = [FEATURES[int(name[1:])] for name in gain.index]
gain.sort_values(ascending=False).round(1).to_frame("gain")

## 6. Register the model in Snowflake

The **Model Registry** stores the trained pipeline inside Snowflake as a versioned object (`WC_ML.MODELS.WC_LITIGATION_XGB`).
It records:

* the model itself, plus the Python packages it needs
* an input/output **signature**, worked out from sample rows: which columns go in and which come out
* metrics and a comment, so you can tell versions apart later

Once registered, the model can be called from Python (`mv.run`) or **from plain SQL**. That makes it usable by
anyone with the right role, without a Python environment. `deploy/04_deploy_scoring.sql` uses the SQL route.

Each run creates a new **version** named with the date and time. Setting `default` chooses which version SQL
uses when no version is named. Rolling back means pointing `default` at an older version.

> **Snowflake vs. local:** locally you'd save the pipeline with `joblib.dump(xgb_pipe, "model.pkl")` and then
> have to manage the file yourself: where it lives, who can load it, which version is current, and which package
> versions it needs. You might use a tool like MLflow for that. The registry handles all of it, and inference runs
> next to the data, so no data has to leave Snowflake.

In [ ]:
from datetime import datetime

from snowflake.ml.registry import Registry

# Connect to the registry in the WC_ML.MODELS schema
reg = Registry(session=session, database_name="WC_ML", schema_name="MODELS")
# A unique version name per run, e.g. V_20261012_153000
version = "V_" + datetime.now().strftime("%Y%m%d_%H%M%S")
xgb_test = test_metrics.set_index("model").loc["xgboost (test)"]

mv = reg.log_model(
    xgb_pipe,                              # the fitted pipeline: preparation + XGBoost
    model_name=MODEL_NAME,
    version_name=version,
    # Example rows the registry uses to work out the input/output signature (column names and types)
    sample_input_data=X_train.head(500),
    # The pipeline needs both libraries at inference time. They are left unpinned because the
    # notebook's pip-installed versions may not exist in the Anaconda channel that warehouse
    # inference uses; the registry picks compatible versions from that channel.
    conda_dependencies=["scikit-learn", "xgboost"],
    # Stored with this version, so you can compare versions later
    metrics={
        "test_year": TEST_YEAR,
        "test_roc_auc": float(xgb_test.roc_auc),
        "test_pr_auc": float(xgb_test.pr_auc),
        "test_brier": float(xgb_test.brier),
        "features": FEATURES,
    },
    comment="Probability a NY workers' comp claim gets attorney representation, using intake-time features.",
)
# Make this the version that SQL calls use by default
reg.get_model(MODEL_NAME).default = version
print("Registered", MODEL_NAME, version)

In [ ]:
# The methods the registered model exposes (predict, predict_proba, ...), with their input
# columns in order and the names of their output columns. Useful when writing the SQL calls.
mv.show_functions()

In [ ]:
# Smoke test: score 10 claims from 2023 (never seen in training) INSIDE Snowflake.
# [Snowflake vs. local] mv.run() takes a Snowpark DataFrame, so the scoring runs in the warehouse
# next to the data and nothing is downloaded. Locally you'd call xgb_pipe.predict_proba(pandas_df).
new_claims = session.table(FEATURE_VIEW).filter(F.col("ASSEMBLY_YEAR") == 2023).limit(10)
mv.run(new_claims, function_name="predict_proba").to_pandas()

In [ ]:
# Every model in WC_ML.MODELS, with its versions and default version
reg.show_models()

## What's next

Open `deploy/04_deploy_scoring.sql` in a SQL worksheet. It calls the model from SQL, creates a procedure that
scores new claims in batches, and schedules it with a Task.

**Running this as a `.py` script instead of a notebook:** the modeling code stays the same. Two things change.
A script doesn't display a cell's last value automatically, so wrap those lines in `print(...)`. And the session
comes from wherever the script runs: `get_active_session()` inside Snowflake, or `Session.builder...create()` on
a laptop. See the README for details.